# Place finder

Search the geographic catalog, inspect identifiers used by forecast endpoints, and map the matching locations.

This live demonstration uses public meteo@uniparthenope data. Values are generated
automatically and are intended for research, education, and general information—not for official
warnings, navigation, or safety-critical decisions. Times are UTC. Run and edit the cells in this
browser-based workspace; your changes stay in your browser unless you download the notebook.


In [ ]:
import json
import os
import sys
from urllib.parse import urlencode
from datetime import datetime

if sys.platform == "emscripten":
    import piplite
    await piplite.install("ipywidgets")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

API_BASE = "https://api.meteo.uniparthenope.it" if sys.platform == "emscripten" else os.environ.get(
    "METEO_API_BASE_URL", "https://api.meteo.uniparthenope.it"
).rstrip("/")
BRAND = "#1f4e79"
ACCENT = "#2a91eb"

def api_json(path, params=None):
    url = f"{API_BASE}/{path.lstrip('/')}"
    if params:
        url = f"{url}?{urlencode(params)}"
    if sys.platform == "emscripten":
        from pyodide.http import open_url
        return json.load(open_url(url))
    import requests
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()

def forecast_frame(place):
    payload = api_json(f"products/wrf5/timeseries/{place}")
    if payload.get("result") != "ok":
        raise ValueError(f"No WRF forecast is available for {place}")
    frame = pd.DataFrame(payload["timeseries"])
    frame["time"] = pd.to_datetime(frame["dateTime"], format="%Y%m%dZ%H%M", utc=True)
    return frame.set_index("time")

PLACES = {"Gulf of Naples": "ca001", "Gulf of Salerno": "ca002", "Cilento": "ca003"}


In [ ]:
query = widgets.Text(value="Napoli", description="Search:", continuous_update=False)
search = widgets.Button(description="Find places", button_style="primary", icon="search")
output = widgets.Output()

def render(_=None):
    with output:
        clear_output(wait=True)
        try:
            places = api_json(f"places/search/byname/{query.value.strip()}")
            rows = []
            for item in places[:50]:
                lon, lat = item.get("pos", {}).get("coordinates", [np.nan, np.nan])
                name = item.get("long_name", {}).get("it") or item.get("name", {}).get("it", "")
                rows.append({"identifier": item.get("id"), "name": name, "longitude": lon, "latitude": lat})
            frame = pd.DataFrame(rows).dropna(subset=["longitude", "latitude"])
            display(frame)
            if not frame.empty:
                fig, ax = plt.subplots(figsize=(8, 5))
                ax.scatter(frame.longitude, frame.latitude, color=ACCENT, edgecolor=BRAND)
                for row in frame.head(12).itertuples():
                    ax.annotate(row.identifier, (row.longitude, row.latitude), xytext=(4, 4), textcoords="offset points")
                ax.set(xlabel="Longitude", ylabel="Latitude", title=f"Matches for {query.value!r}")
                ax.grid(alpha=.25)
                plt.show()
        except Exception as exc:
            display(HTML(f"<b>Search failed:</b> {exc}"))

search.on_click(render)
query.on_submit(render)
display(widgets.VBox([widgets.HBox([query, search]), output]))
render()
